## Dataset Selection: FATURA for Invoice Information Extraction

### Overview

For the invoice information extraction task, we selected **FATURA** (Limam, Dhiaf & Kessentini, 2023) as the base corpus for building our training data. FATURA is a synthetic, multi-layout invoice image dataset comprising 10,000 images spanning 50 distinct templates (200 images per template), with 24 annotated semantic classes covering bounding box coordinates, text content, and object class. Rather than adopting the dataset's native annotation scheme directly, we use FATURA's images as raw input and re-annotate them with a larger teacher model, whose outputs then serve as ground truth for fine-tuning a smaller production model. This section documents the dataset-level decisions that informed this pipeline: the choice between the two published dataset versions, the relevance of the background-color augmentation FATURA offers, and the sampling strategy across templates and images.

### FATURA v1 vs. v2

FATURA has been released in two versions on Zenodo: the original release (record 8261508) and a second release published in December 2023 under the name **FATURA2** (record 10371464). Both versions share the same underlying generation process: 50 hand-designed templates, each instantiated 200 times through randomized field content and position, with annotations provided in three formats (a native FATURA format, COCO, and a HuggingFace-compatible token format).

The publicly documented difference between the two versions is twofold:

1. **An additional image set with colored backgrounds.** FATURA v2 supplies a second 10,000-image set generated from the same 50 templates and the same underlying field content, but rendered with template-specific background colors (consistent across all instances of a given template) rather than plain white. This effectively doubles the number of rendered images available (20,000 total across both background variants) without introducing new templates, classes, or field content.
2. **Formalized benchmark splits.** v2 explicitly ships the train/validation/test splits used for the two evaluation protocols proposed in the accompanying paper — an intra-template split (all templates present in every split, testing generalization to unseen content) and an inter-template split (templates held out entirely, testing generalization to unseen layouts).

Critically, the **annotation schema and label content are unchanged between versions** — the same 24 classes, the same bounding-box/text/class structure, applied to the same underlying template population. We manually verified this directly on our copy of the dataset: annotations for v1 and v2 are identical at the instance level, differing only in which background-color variant of a template they accompany. This confirms that v2 is best understood as an image-rendering extension of v1 rather than a re-annotation or schema revision.

### Relevance of the Colored-Background Variant to Our Use Case

Because the v1/v2 difference is confined to background color and does not alter labels, its practical value depends entirely on the deployment context. FATURA's authors motivate the colored-background set as an augmentation to improve robustness to visual noise and stylistic variation in invoice layouts.

For our production setting, this augmentation axis is not relevant: incoming documents are clean, white-background PDFs rather than scanned or stylistically varied invoices. Training or evaluating on the colored-background variant would introduce a distribution of visual noise that our production input distribution does not contain, without any corresponding benefit — since the labels themselves are identical, the colored set adds pixel-level variation but no new semantic, layout, or field-boundary information. We therefore restrict our pipeline to the white-background image set (common to both v1 and v2) and treat the two dataset versions as interchangeable for annotation purposes, selecting whichever is more convenient to source without loss of information.

### Why Reducing Images per Template Does Not Improve Generalization

A separate question raised during scoping was whether reducing the number of images sampled per template (e.g., using fewer than the full 200 per template) would reduce overfitting and improve generalization, given the dataset's structure of 50 templates × 200 images. We find this not to be a productive lever, for the following reasons.

**Overfitting in FATURA is a template-axis phenomenon, not an image-count axis phenomenon.** The dataset's own evaluation methodology draws exactly this distinction. FATURA defines two benchmark protocols:

- An **intra-template split**, where images from every template are distributed across train/validation/test. This measures a model's ability to generalize to new *content* rendered on *already-seen* layouts.
- An **inter-template split**, where entire templates are held out of training and reserved for testing. This measures a model's ability to generalize to *unseen layouts* entirely.

The second protocol is the one that actually stresses layout generalization — and it is controlled by how many distinct *templates* the model is exposed to, not by how many images per template it sees. Reducing images per template while keeping all 50 templates in the training set does not remove any templates from the model's exposure; it only reduces the amount of content and positional variation the model observes for each layout it already sees.

**Reducing images per template can be counterproductive.** Within a single FATURA template, the 200 instances exist specifically to vary field text and field position around a fixed layout skeleton. This variation is what discourages the model from memorizing fixed bounding-box coordinates for a given class within a template and instead forces it to key off more transferable signals (field semantics, relative position, nearby anchor text). Cutting the number of images per template shrinks this within-template variation, which — if anything — increases the risk of the model latching onto template-specific coordinate patterns, the opposite of the desired effect.

**The correct lever for layout generalization is template diversity, not per-template sampling density.** If the goal is a model that generalizes beyond the 50 FATURA layouts to invoice formats not represented in the dataset, the effective interventions are:

- Evaluating (and, if feasible, training) using the inter-template split rather than the intra-template split, to get an honest read on layout generalization.
- Expanding the set of distinct templates the model is exposed to — either by supplementing FATURA with invoices from other sources/layouts, or by using the teacher model to annotate additional out-of-distribution invoice layouts beyond FATURA's 50.
- Keeping images per template high enough to preserve the intra-template content/position variation FATURA was designed to provide, since this is what prevents coordinate memorization within a layout.

> Which is also not our goal, because in production we will currently be working with only one template.

**Conclusion.** We retain the full 200 images per template across all 50 templates. Reducing images per template would reduce the within-template content variation that discourages coordinate memorization, while doing nothing to address the actual driver of layout overfitting — template count.

## Preprocessing Decision: Fixed Image Sizing for Qwen-VL Teacher Annotation

### Current State

The current preprocessing step performs no explicit resizing:

```python
image = Image.open(img_path).convert("RGB")
data_url = image_to_data_url(image)
stem = img_path.stem
```

Each image is therefore sent to the Qwen-VL teacher model at its native resolution. When using a self-hosted Qwen model through vLLM, the image is subsequently processed by the model's Hugging Face image processor, which performs its own resolution adjustment according to the model's vision-token constraints.

For this project, where Qwen-VL is being used as the teacher to generate the annotations that will later train a smaller production model, the question is whether we should control this resizing explicitly or rely on Qwen's native preprocessing.

### Why Resolution Control Matters

There are three relevant considerations for this dataset-generation stage:

1. **Reproducibility of the generated labels.**
   The teacher's output becomes the training ground truth for the student model. Therefore, the visual input provided to the teacher should be deterministic and versionable. Relying entirely on implicit preprocessing makes the exact image representation less visible from the dataset-building pipeline.

2. **Vision-token and inference cost.**
   Qwen-VL models do not process an arbitrary number of image pixels directly. The image is transformed into visual tokens, and the number of tokens increases with image resolution. Sending very large invoice images therefore increases the multimodal input size and can increase GPU memory usage, latency, and overall inference cost.

3. **OCR and document-reading fidelity.**
   The opposite extreme is also undesirable. Invoices contain small text, addresses, reference numbers, totals, and table information. Reducing the image too aggressively can remove information that the teacher needs to produce accurate structured labels.

For teacher annotation, the objective should therefore **not** be to minimize image resolution as much as possible. The objective is to provide Qwen with enough resolution to reliably read the document while avoiding pixels that exceed the model's effective vision-token budget.

### Qwen-VL's Native Image Processing

For Qwen2.5-VL, image resizing is part of the model's multimodal preprocessing pipeline. The Qwen processor uses a pixel-based constraint rather than simply requiring a fixed width or height.

Conceptually, the processing can be expressed as:

```text
Original image
      ↓
Qwen image processor
      ↓
Smart resize / pixel constraint
      ↓
Patch-based vision encoder
      ↓
Visual tokens
      ↓
Qwen-VL
```

The important parameters are the **minimum and maximum number of pixels**, rather than a universal fixed long-edge value.

When Qwen2.5-VL is served with vLLM, these preprocessing parameters can be controlled through the multimodal processor configuration, for example:

```bash
--mm-processor-kwargs '{
    "min_pixels": ...,
    "max_pixels": ...
}'
```

This allows the image processor to perform the resize while respecting the dimensions and patch structure expected by the Qwen vision encoder.

This is preferable to introducing an arbitrary PIL resize such as:

```python
MAX_LONG_EDGE = 1568
```

because a fixed long-edge rule is not necessarily aligned with Qwen's actual visual-token budget or patch-grid requirements.

### Recommended Approach for Qwen2.5-VL

For this project, **we won't introduce a generic fixed long-edge resize such as 1568 px**. That value comes from other vision-model/API configurations and is not the appropriate sizing criterion for Qwen2.5-VL.

Instead, the preferred approach is:

> **Let Qwen's native image processor perform the resolution adjustment, while explicitly controlling and recording its pixel/token budget.**

For a self-hosted teacher running through vLLM, configure `min_pixels` and `max_pixels` according to the Qwen2.5-VL model being used and the available GPU memory.

For example, conceptually:

```bash
vllm serve Qwen/Qwen2.5-VL-32B-Instruct \
    --mm-processor-kwargs '{
        "min_pixels": ...,
        "max_pixels": ...
    }'
```

The exact values should be selected experimentally based on:

* teacher annotation accuracy,
* visual token count,
* GPU memory consumption,
* throughput,
* and the smallest text that must remain readable in the invoices.

The important point is that the **Qwen processor should remain responsible for the final resize**, rather than replacing it with an arbitrary PIL-based long-edge resize.

### Should the Images Be Resized Before Sending Them to vLLM?

For the current Qwen-based annotation pipeline, there are two possible approaches.

#### Option A — Let Qwen/vLLM preprocess the images

```text
Original image
      ↓
vLLM
      ↓
Qwen image processor
      ↓
Native Qwen smart resize
      ↓
Vision encoder
```

This is the recommended default.

It preserves Qwen's native preprocessing behavior and ensures that the resulting dimensions are compatible with the model's patch/tokenization requirements.

It also avoids maintaining a second resizing implementation that could potentially differ from the preprocessing used when the student model is later trained.

#### Option B — Resize manually with PIL

```text
Original image
      ↓
PIL resize
      ↓
vLLM
      ↓
Qwen image processor
      ↓
Vision encoder
```

This can be useful if the original documents are extremely large and we want to eliminate obviously unnecessary input pixels before they reach the inference server.

However, a generic:

```python
image.resize(...)
```

should **not** be used with an arbitrary fixed long-edge value without first considering Qwen's pixel budget and patch size.

Otherwise, the image may be resized once by PIL and then resized again by the Qwen processor, producing unnecessary preprocessing and potentially reducing document detail.

### Teacher Annotation vs. Student Training

This distinction is particularly important for this project.

The teacher annotation stage is designed to generate high-quality ground truth from approximately 10,000 invoice images. Therefore, the priority should be:

```text
Annotation quality
        ↓
Sufficient visual resolution
        ↓
Controlled inference cost
        ↓
Maximum throughput
```

We should not aggressively reduce resolution simply because fewer visual tokens are cheaper.

A small reduction in teacher accuracy can propagate into thousands of training examples and ultimately affect the student model.

For the student model, however, the situation can be different. Once the production architecture is selected, its own image-resolution budget can be optimized based on the target hardware, latency requirements, and extraction accuracy.

Therefore:

> **Teacher preprocessing should be optimized primarily for label quality; production/student preprocessing should be optimized for the final accuracy–latency–cost trade-off.**

### Final Recommendation

For the current Qwen-based teacher annotation pipeline:

* **Do not use the Claude/GPT/Gemini resolution limits as the basis for preprocessing.**
* **Do not use an arbitrary fixed long-edge value such as 1568 px.**
* Keep the original images intact as the source dataset.
* Use **Qwen2.5-VL's native image processor** for resizing.
* When serving through vLLM, control the processor using **`min_pixels` / `max_pixels`** rather than implementing an independent long-edge rule.
* Choose the maximum pixel budget experimentally, prioritizing teacher annotation quality while monitoring GPU memory, visual-token count, and throughput.
* If the source images are exceptionally large, a preliminary resize can be considered, but it should be based on Qwen's actual pixel/token constraints rather than an unrelated API's resolution ceiling.

In short, the appropriate principle for this project is:

> **Do not force Qwen2.5-VL into a generic fixed image size. Control its native pixel budget and let the Qwen processor perform the model-compatible resize.**

This gives us the best balance between reproducibility, document readability, and inference efficiency while remaining aligned with the preprocessing used by the Qwen-VL architecture itself.


# Structured Output Enforcement with Guided Decoding

## Objective

The teacher model must generate structured JSON from invoice images. Since these annotations will be used as training data for the smaller production model, the output format must be consistent and reliable.

The recommended approach separates three responsibilities:

1. **Prompt schema:** communicate the expected fields and their meaning to Qwen.
2. **Guided decoding:** enforce the JSON structure during generation using vLLM.
3. **Post-processing:** validate and handle exceptional outputs.

## 1. Schema in the Prompt

The schema included in the prompt should be a **compact TypeScript-style representation** rather than a full JSON Schema.

For example:

```typescript
{
  fields: {
    supplier_name: string | null,
    supplier_phone_number: string | null,
    supplier_address: Address | null,
    customer_name: string | null,
    customer_address: Address | null,
    invoice_number: string | null,
    document_type: "invoice" | "tax_invoice" | null,
    date: string | null,
    due_date: string | null,
    period: string | null,
    total_net: number | null,
    total_tax: number | null,
    total_amount: number | null,
    taxes: Tax[],
    line_items: LineItem[]
  }
}
```

This schema is only for the model to understand the expected fields and value types.

The prompt should still contain all semantic extraction instructions, such as:

* distinguishing supplier from customer;
* identifying the correct invoice, due, and period dates;
* mapping invoice-table columns to line-item fields;
* handling missing or ambiguous information.

These semantic rules cannot be enforced by JSON Schema alone.

## 2. Guided Decoding with vLLM

The actual schema enforcement should be handled by **vLLM guided decoding**.

The JSON Schema is not passed to `vllm serve`. Instead, it is provided with each inference request through `extra_body`.

A formal JSON Schema can be generated from Pydantic:

```python
from pydantic import BaseModel

class InvoiceRecord(BaseModel):
    fields: InvoiceFields

GUIDED_SCHEMA = InvoiceRecord.model_json_schema()
```

Then:

```python
response = await client.chat.completions.create(
    model=served_model,
    messages=[...],
    max_tokens=max_new_tokens,
    temperature=0,
    extra_body={
        "guided_json": GUIDED_SCHEMA
    },
)
```

This constrains the generated output to the specified structure and types during decoding.

Therefore, the prompt schema and the guided schema are **two separate artifacts with different purposes**:

* **TypeScript schema → helps Qwen understand the expected output.**
* **JSON Schema + `guided_json` → constrains what vLLM allows Qwen to generate.**

## 3. Role of `extract_json`

`extract_json` is a custom post-processing function, not a vLLM feature.

Without guided decoding, the pipeline relies on:

```text
Model output → extract_json() → json.loads()
```

With guided decoding, structural correctness is enforced during generation:

```text
Model → guided decoding → JSON → validation
```

Therefore, `extract_json` should no longer be the primary mechanism responsible for obtaining valid JSON.

## 4. Recommended Pipeline

For the Qwen-VL teacher annotation stage, the recommended configuration is:

* **Prompt:** compact TypeScript-style schema + detailed semantic extraction rules.
* **vLLM:** use `guided_json` with a formal JSON Schema generated from Pydantic.
* **Generation:** `temperature=0` for deterministic extraction.
* **Dataset:** store the final annotation and, preferably, the raw model output for auditing.

This provides a clear separation between **semantic instructions**, **structural enforcement**, and **post-generation validation**.

### Final Decision

The teacher annotation pipeline should use **guided decoding as the primary structural guarantee**, rather than relying on prompt instructions and `extract_json` alone.

> **The prompt tells Qwen what the fields mean; guided decoding enforces the structure; post-processing validates and audits the result.**


# Moving to Offline vLLM Inference

## 1. Proposed Change

The current annotation pipeline uses Qwen-VL through a vLLM server and an OpenAI-compatible `AsyncOpenAI` client. The proposed approach is to use vLLM's **offline Python API** directly through `vllm.LLM()`.

The main motivation is to remove the HTTP serving layer and associated request-processing overhead while keeping vLLM's GPU scheduling and continuous batching.

## 2. What Changes

The current inference call:

```python
response = await client.chat.completions.create(
    model=served_model,
    messages=[...],
    ...
)
```

is replaced by the offline API:

```python
outputs = llm.chat(
    requests,
    sampling_params=sampling_params
)
```

The following components are therefore removed:

* `AsyncOpenAI`
* HTTP communication with the vLLM server
* JSON serialization/deserialization for API requests
* Base64/data-URL image encoding
* `asyncio.Semaphore` used to control API concurrency
* Server-related CLI parameters such as `--server-url` and `--api-key`

The existing annotation logic remains unchanged, including:

* image discovery and manifest handling;
* prompt and target schema;
* JSON extraction and validation;
* output saving;
* failure tracking;
* throughput measurement.

## 3. Batching and Performance

Moving to offline inference does **not** introduce continuous batching. vLLM already performs continuous batching in the current server-based architecture.

The expected benefit comes instead from removing the overhead between the application and the inference engine:

* HTTP communication;
* request serialization and parsing;
* Base64 image encoding;
* server-side request handling;
* client-side concurrency management.

The GPU computation itself remains essentially unchanged for the same model, image resolution, quantization, prompt, and generation parameters. Therefore, offline inference should not be presented as a way to reduce model FLOPs.

The actual performance improvement must be measured experimentally because the current server runs locally. In this situation, HTTP and serialization overhead may represent only a small fraction of the approximately 9.75 s/document baseline observed on the T4.

## 4. Chunked Offline Inference

The 10,000 invoices should not be loaded into a single Python list before inference.

Instead, the dataset should be processed in chunks, for example:

```text
32–128 requests per chunk
```

Results should be written incrementally after each chunk.

This limits CPU memory consumption while still allowing vLLM to efficiently schedule the submitted requests.

## 5. Multimodal Considerations

The migration requires adapting the request format to vLLM's native multimodal offline API.

For Qwen-VL, each request must contain both the text prompt and its corresponding image using the multimodal input format expected by the model.

The existing image-to-data-URL conversion used by the OpenAI-compatible API can therefore be removed if the image is provided directly in the native offline request format.

This is the main implementation point that must be validated during the migration.

## 6. Validation

The migration should first be benchmarked on a representative subset before running the complete dataset.

The same documents, model, prompt, image preprocessing, generation parameters, and hardware should be used for both approaches.

The following should be measured:

* total processing time;
* documents per second;
* average time per document;
* image preprocessing time;
* inference time;
* GPU utilization;
* peak GPU memory usage.

The benchmark should first be performed on the current environment and then repeated on the target rented GPU (A100/H100).

The comparison should also verify that the offline implementation produces equivalent extraction results to the existing pipeline.

## 7. Conclusion

Offline vLLM inference is a **serving-layer optimization**, not a model or batching optimization.

The main advantages are the removal of HTTP, serialization, Base64 encoding, and client-side concurrency management while retaining vLLM's continuous batching and GPU execution.

Because the current server operates locally, the expected speedup cannot be assumed to be large. A controlled benchmark is therefore required before migrating the full 10,000-document annotation run.


# Automatic Prefix Caching (APC)

## 1. Overview

Automatic Prefix Caching (APC) is a vLLM engine feature that reuses previously computed KV-cache states for requests sharing an identical prefix.

For the current Qwen-VL annotation workload, requests share the same prompt and schema while the invoice image changes. This makes APC a potentially useful optimization because the common textual prefix may be reused across requests.

APC is independent of the API interface: it can be used both with the current OpenAI-compatible vLLM server and with the offline `vllm.LLM()` API.

## 2. Configuration

With the current server-based architecture, APC can be enabled with:

```bash
vllm serve ... --enable-prefix-caching
```

No changes to the `AsyncOpenAI` inference call are required.

With offline vLLM, the equivalent configuration is:

```python
from vllm import LLM

llm = LLM(
    model=MODEL_ID,
    enable_prefix_caching=True,
    ...
)
```

In both cases, prefix caching is handled internally by vLLM; no application-level cache implementation is required.

## 3. Applicability to the Qwen-VL Workload

The annotation requests have a common prompt but different images. Therefore, the potential reusable portion is primarily the shared textual context.

However, multimodal inputs require careful benchmarking. The image is processed through Qwen-VL's multimodal pipeline, and identical text alone does not imply that the entire request will benefit equally from prefix caching.

Consequently, APC should be evaluated using the actual Qwen-VL workload rather than assuming a specific cache-hit rate or speedup.

The request structure should also not be changed solely for APC purposes without benchmarking, since Qwen-VL's multimodal preprocessing determines how the image and text are represented internally.

## 4. Expected Impact

APC does **not**:

* reduce the model size;
* reduce the number of generated tokens;
* replace continuous batching;
* eliminate image processing;
* require changes to the extraction prompt.

Its potential benefit comes from avoiding repeated computation for identical request prefixes.

Because the current workload contains 10,000 invoices with a shared prompt but unique images, the magnitude of the benefit depends on how much of the multimodal computation is actually reusable.

## 5. Evaluation

APC should therefore be treated as a low-cost optimization and benchmarked independently.

A representative test should compare:

| Configuration                    | Purpose                        |
| -------------------------------- | ------------------------------ |
| vLLM + continuous batching       | Baseline                       |
| vLLM + continuous batching + APC | Measure prefix-cache benefit   |
| Offline vLLM + APC               | Evaluate combined architecture |

The comparison should use the same model, prompts, images, generation parameters, and hardware, while measuring:

* total processing time;
* documents/second;
* GPU utilization;
* GPU memory usage;

## Conclusion

APC is straightforward to enable in both architectures and does not require a significant code change. It should therefore be tested as an incremental optimization.

The main architectural change remains the transition to offline vLLM; **APC is an independent engine-level optimization that can be enabled before or after that migration.**


## Using FATURA Annotations as a Verification and Validation Layer

### Purpose

Beyond serving as raw images for the teacher-model annotation pipeline, FATURA's own ground-truth labels can be repurposed as an independent check, in two distinct roles:

1. **A pre-selection benchmark** (200–500 images): compare candidate teacher models (NuExtract3, Qwen3-VL-8B, Qwen3-VL-32B) against FATURA's known labels to get a quick read on extraction quality before committing to a full 10k-image annotation run with one of them.
2. **A partial validation pass on the full 10k run**: once the chosen teacher has annotated the complete dataset, spot-check its output against FATURA's own labels wherever FATURA happens to provide a value for that field, as a cheap sanity signal that doesn't require additional manual labeling.

The key working assumption, as observed during manual review, is that FATURA's labels are **not incorrect, only incomplete and inconsistently formatted** — fields are sometimes missing or the annotated text includes the on-page label text and formatting verbatim (e.g. `"TOTAL : 734.33 EUR"` rather than a clean `734.33`). This shapes both which annotation format to use and how comparisons should be scored.

### Which of the Three Formats to Use

FATURA ships three annotation formats per the accompanying paper: its own **native/custom format**, a **COCO format**, and a **HuggingFace format built specifically for LayoutLMv3-style token classification**. For this verification use case, the native format is the clear choice.

- **Native format** — already structured as `field_name → {"bbox": ..., "text": ...}`, i.e. exactly the key/value shape we want to diff against a VLM's structured JSON output. As shown in the example, a value like `TOTAL` maps directly to `{"bbox": [...], "text": "TOTAL : 734.33 EUR"}`. No reconstruction is needed to get a per-field string.
- **COCO format** — organizes annotations as a flat list of `{image_id, category_id, bbox, ...}` objects, requiring a join against a separate `categories` list to recover the class name, and a group-by on `image_id` to reassemble all fields for one document. Even after that work, you end up back at roughly the same field→text mapping the native format already gives you directly — so COCO adds parsing overhead without adding information for this task. It would be the right choice if we were training or benchmarking an object-detector, which we are not.
- **HuggingFace/LayoutLMv3 format** — token-level: individual words, their bounding boxes, and BIO-style tags. This is the worst fit for our purpose, since reconstructing a full field value (e.g. a three-line address or a payment-details block) requires grouping contiguous same-tag tokens back into a string, correctly reinserting line breaks and spacing, and handling reading-order edge cases. That reconstruction effort buys nothing over just reading the native format's pre-assembled `text` field.

**Parse the native format directly.** It requires the least code, introduces no reconstruction risk, and matches the shape of the VLM's own structured output field-for-field.

### Extracting Clean Field/Value Pairs

A few characteristics of the native format need explicit handling, visible directly in the sample annotation:

- Most semantic fields (`BUYER`, `DATE`, `DUE_DATE`, `TOTAL`, `TAX`, etc.) are dicts with `bbox` and `text`.
- A handful of keys are **not** usable field values and should be excluded from field-level comparison:
  - `TABLE` — a nested list of bounding boxes for the line-items region, with no per-item text at this level.
  - `INVOICE_INFO` — frequently an empty container.
  - `OTHER` — a catch-all dump that tends to duplicate most of the page's text; it's not a discrete field and would corrupt precision if treated as one.
  - `LOGO` — bbox only, no text (it's a graphical element, not a text field).
- Field `text` values are **not normalized** — they retain the on-page label and formatting exactly as rendered (`"PO Number :35"`, `"Date: 20-Mar-2008"`, `"SUB_TOTAL : 725.30 EUR"`), and some are genuinely multi-line blocks (`BUYER`, `SELLER_ADDRESS`, `PAYMENT_DETAILS`, `NOTE`, `TOTAL_WORDS`). This means exact-string comparison against a VLM's cleaned extraction (e.g. `"734.33"`) will almost always fail even when the VLM is correct — the comparison needs to be tolerant of formatting, not exact.

Conceptually, the extraction is a single pass over the top-level keys of the JSON: keep only entries that are dicts containing a non-empty `text`, and drop the excluded keys above. For example:

```python
EXCLUDE_KEYS = {"TABLE", "INVOICE_INFO", "OTHER", "LOGO"}
# for key, value in raw_annotation.items():
#     if key not in EXCLUDE_KEYS and "text" in value:
#         fields[key] = value["text"].strip()
```

This gives, per document, a dict of only the fields FATURA actually populated — which is exactly what's needed given the "missing but not false" assumption: absent fields are simply not checked, rather than being treated as ground-truth negatives.

### Comparing Against VLM Output

Because FATURA's field text is unnormalized, comparisons should use **containment or fuzzy matching rather than exact equality**:

Conceptually: lowercase and whitespace-normalize both strings, then check containment first, falling back to a fuzzy similarity ratio (e.g. via `difflib.SequenceMatcher`) above some threshold (0.6 is a reasonable starting point, but worth tuning against a hand-checked sample):

```python
# v, f = normalize(vlm_value), normalize(fatura_text)
# match = (v in f) or (SequenceMatcher(None, v, f).ratio() >= threshold)
```

A substring check (`v in f`) handles the common case directly — the VLM's clean value is a literal substring of FATURA's labeled text once both are lowercased and whitespace-normalized (this covers `TOTAL`, `TAX`, `DATE`, `DUE_DATE`, `PO_NUMBER`, `SUB_TOTAL`, and similar single-line fields well). The fuzzy-ratio fallback catches near-misses on multi-line or more loosely formatted fields (`BUYER`, `SELLER_ADDRESS`, `PAYMENT_DETAILS`) where exact substring containment is too strict (e.g. the VLM may reformat a multi-line address onto one line, or vice versa).

### Applying This to the Two Use Cases

**1. Teacher-model benchmark pass (200–500 images).** Sample images across all 50 templates (stratified, not just the first N images, to avoid overrepresenting a handful of templates), run each candidate model's extraction, and score per-field match rate against FATURA's available labels:

Conceptually, for each field FATURA provides, check whether the VLM output contains a matching value and record a per-field boolean; aggregate this into a per-field and overall match rate for each candidate (NuExtract3, Qwen3-VL-8B, Qwen3-VL-32B), using an identical image-resolution policy across all three as established in the earlier preprocessing chapter, so the comparison isolates model quality rather than confounding it with input resolution. This gives a fast, quantitative basis for picking the production teacher before spending the budget on the full 10k run.

**2. Validation pass on the full 10k output.** Run the same `extract_fatura_fields` / `field_matches` logic over the teacher's actual annotations for all 10,000 images, but treat this as a **flagging mechanism rather than a ground-truth grading**: for any field FATURA provides but the teacher's output doesn't match, flag that document/field pair for manual spot review rather than assuming the teacher is wrong. Given FATURA's fields are missing/poorly-shaped rather than false, a mismatch usually means one of: the teacher genuinely erred, FATURA's formatting broke the fuzzy match (worth re-checking the threshold), or the teacher's output is actually more correct than FATURA's synthetic label. This pass is best used to **surface a manageable review queue** (e.g. the worst-scoring few hundred documents or fields) rather than to compute a hard accuracy number for the full 10k set, since FATURA doesn't cover every field on every document and can't fully substitute for manual review.

### Alternative: Cleaning the Annotations Instead of Fuzzy-Matching Them

The comparison approach above works around FATURA's label-prefixed text at match time. The alternative is to clean the annotations once, upfront, and produce a normalized `field → value` version of the dataset that's directly comparable to a VLM's structured output without any fuzzy logic at comparison time. This is worth doing, with two caveats.

**Where it works cleanly.** Several fields consistently follow a `LABEL : value` or `Label: value` pattern on a single line — `TOTAL`, `SUB_TOTAL`, `TAX`, `DISCOUNT`, `DATE`, `DUE_DATE`, `PO_NUMBER` in the sample all fit this shape. For these, a per-field regex that strips everything up to and including the first colon (then trims whitespace and any trailing unit/currency token if you want pure numeric values) reliably recovers a clean value:

```python
# re.sub(r"^[^:]*:\s*", "", fatura_text).strip()
# e.g. "TOTAL : 734.33 EUR" -> "734.33 EUR" -> optionally "734.33" with a further currency strip
```

**A better framing: calibrate per template, not globally.** A single rule meant to generalize across all 50 templates' differing label wording is genuinely hard to write and fragile, as above. But that's the wrong unit of generalization here — FATURA gives us the template identity for every image (via the filename structure), and within one template, the label text and formatting are pixel-identical across all 200 instances — only the field *content* changes. That's a much stronger, more exploitable constraint than trying to handle 50 different phrasings with one rule: instead, calibrate one small extraction rule per (template, field) pair, using the 200 available samples of that template as calibration data, and apply it only to images from that template.

**Deriving the rule automatically instead of by hand.** Because the label portion is constant across a template's 200 samples and the value portion varies, the fixed/variable split can be induced algorithmically rather than hand-written: align the raw `text` strings for a given field across all instances of a template, and treat character positions where every sample agrees as fixed label/formatting, and positions where samples diverge as the value slot. This effectively reverse-engineers a fill-in-the-blank template per field per template, with no manual regex authoring needed for any of the 50 templates individually.

**One real wrinkle.** This isn't always a clean "prefix is fixed, suffix is the value" split. Some fields embed a variable value *inside* what otherwise looks like label text — in the sample, `"DISCOUNT(1.85%): (-)  13.42"` and `"TAX:VAT (3.88%):  28.18 EUR"` both have a percentage that varies in the middle of the string, before the actual total/tax value at the end. A simple longest-common-prefix approach breaks as soon as the percentage digits start disagreeing across samples. The induction needs to compare *all* aligned positions across the samples (not stop at the first point of disagreement), so it can recover a pattern with multiple variable slots — e.g. discount percentage and discount amount as two separate captured values — rather than assuming there's only one variable region at the end.

**Conclusion.** Per-template calibrated alignment provides a reliable extraction method for clearly labeled fields such as `TOTAL`, `SUB_TOTAL`, `TAX`, `DISCOUNT`, `DATE`, `DUE_DATE`, and `PO_NUMBER`, enabling exact-value comparisons during validation. For multi-line fields such as `BUYER` and `SELLER_ADDRESS`, the approach remains promising but should be manually spot-checked across representative templates before being applied at scale, as line wrapping may vary with the content.